In [ ]:
import warningsimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltfrom statsmodels.tsa.arima.model import ARIMAfrom sklearn.preprocessing import StandardScalerfrom pmdarima import auto_arimawarnings.filterwarnings("ignore")def load_data(url):    df = pd.read_csv(url)    df['date'] = pd.to_datetime(df['date'])    df.set_index('date', inplace=True)    df = df.resample('h').mean().asfreq('h')    df['values'] = df['values'].interpolate()    return df, StandardScaler()def forecast_arima(data, order, steps=48, confidence=0.95):    model = ARIMA(data, order=order).fit()    forecast_result = model.get_forecast(steps=steps)    forecasts = forecast_result.predicted_mean    conf_int = forecast_result.conf_int(alpha=1 - confidence)    return forecasts, conf_int.iloc[:, 0], conf_int.iloc[:, 1]def bootstrap_ci(model_order, data, steps=48, n_bootstraps=100, confidence=0.95):    forecasts = []    for i in range(n_bootstraps):        try:            sample = data.sample(n=len(data), replace=True).sort_index()            model = ARIMA(sample, order=model_order).fit()            forecasts.append(model.forecast(steps=steps).values)        except:            continue    if not forecasts:        raise RuntimeError("All bootstrap iterations failed")    forecasts = np.array(forecasts)    alpha = (1 - confidence) / 2    return (np.mean(forecasts, axis=0),            np.percentile(forecasts, alpha * 100, axis=0),            np.percentile(forecasts, (1 - alpha) * 100, axis=0))def plot_forecast(historical, test, forecasts, lower, upper, title=""):    fig, ax = plt.subplots(figsize=(12, 5))    ax.plot(historical.index, historical.values, 'k-', alpha=0.3, linewidth=0.8)    ax.plot(test.index, test.values, 'g-', linewidth=1.5, label='Actual')    ax.plot(test.index, forecasts, 'r-', linewidth=1.5, label='Forecast')    ax.fill_between(test.index, lower, upper, color='r', alpha=0.15)    ax.axvline(test.index[0], color='k', linestyle='--', linewidth=0.8, alpha=0.5)    ax.set_xlabel('Date')    ax.set_ylabel('Value')    if title:        ax.set_title(title)    ax.legend(frameon=False, loc='best')    ax.spines['top'].set_visible(False)    ax.spines['right'].set_visible(False)    plt.tight_layout()    plt.show()url = "https://raw.githubusercontent.com/kylejones200/time_series/refs/heads/main/ercot_load_data.csv"df, scaler = load_data(url)train_raw = df['values'].iloc[:-48]test_raw = df['values'].iloc[-48:]train_scaled = pd.Series(    scaler.fit_transform(train_raw.values.reshape(-1, 1)).flatten(),    index=train_raw.index)test_scaled = pd.Series(    scaler.transform(test_raw.values.reshape(-1, 1)).flatten(),    index=test_raw.index)auto_model = auto_arima(train_scaled, seasonal=False, trace=False,                         suppress_warnings=True, stepwise=True)best_order = auto_model.orderforecasts_scaled, lower_scaled, upper_scaled = forecast_arima(train_scaled, best_order, steps=48)boot_forecasts_scaled, boot_lower_scaled, boot_upper_scaled = bootstrap_ci(    best_order, train_scaled, steps=48, n_bootstraps=50)def inverse_transform(data):    return scaler.inverse_transform(np.array(data).reshape(-1, 1)).flatten()forecasts = inverse_transform(forecasts_scaled)lower = inverse_transform(lower_scaled)upper = inverse_transform(upper_scaled)test_actual = inverse_transform(test_scaled)boot_forecasts = inverse_transform(boot_forecasts_scaled)boot_lower = inverse_transform(boot_lower_scaled)boot_upper = inverse_transform(boot_upper_scaled)test_series = pd.Series(test_actual, index=test_raw.index)plot_forecast(df['values'], test_series, forecasts, lower, upper, "ARIMA Forecast")plot_forecast(df['values'], test_series, boot_forecasts, boot_lower, boot_upper, "Bootstrap Forecast")